In [ ]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


print("Upload your CLASS A CSV:")
uploaded_a = files.upload()

class_a_file = list(uploaded_a.keys())[0]

class_a = pd.read_csv(class_a_file)

print("\nClass A columns:")
print(class_a.columns.tolist())


print("\nUpload your CLASS B CSV:")
uploaded_b = files.upload()

class_b_file = list(uploaded_b.keys())[0]

class_b = pd.read_csv(class_b_file)

print("\nClass B columns:")
print(class_b.columns.tolist())

Upload your CLASS A CSV:


Saving Class_A_Criminal_Usernames_Master_Database.csv to Class_A_Criminal_Usernames_Master_Database.csv

Class A columns:
['Real Name', 'Username', 'Age', 'Sex', 'Nationality', 'Crime', 'Source', 'Source URL', 'Review Status', 'Notes']

Upload your CLASS B CSV:


Saving Class_B_X_YouTube_Final.csv to Class_B_X_YouTube_Final.csv

Class B columns:
['item', 'qid', 'platform', 'property_id', 'handle', 'itemLabel', 'occupations', 'gender', 'citizenships', 'languages', 'date_of_birth', 'age']


In [ ]:
def find_username_column(df):
    possible_names = [
        "username",
        "user_name",
        "handle",
        "user",
        "name"
    ]

    columns_lower = {
        str(col).lower().strip(): col
        for col in df.columns
    }

    for name in possible_names:
        if name in columns_lower:
            return columns_lower[name]

    raise ValueError(
        f"Could not automatically find username column. "
        f"Columns found: {df.columns.tolist()}"
    )


a_username_col = find_username_column(class_a)
b_username_col = find_username_column(class_b)

print("Class A username column:", a_username_col)
print("Class B username column:", b_username_col)

Class A username column: Username
Class B username column: handle


In [ ]:
class_a_clean = class_a[[a_username_col]].copy()
class_a_clean.columns = ["username"]

class_b_clean = class_b[[b_username_col]].copy()
class_b_clean.columns = ["username"]


class_a_clean = class_a_clean.dropna()
class_b_clean = class_b_clean.dropna()


class_a_clean["username"] = class_a_clean["username"].astype(str)
class_b_clean["username"] = class_b_clean["username"].astype(str)


class_a_clean = class_a_clean.drop_duplicates("username")
class_b_clean = class_b_clean.drop_duplicates("username")



class_a_clean["label"] = 1
class_b_clean["label"] = 0


print("Class A usernames:", len(class_a_clean))
print("Class B usernames:", len(class_b_clean))

Class A usernames: 234
Class B usernames: 19750


In [ ]:
n = min(len(class_a_clean), len(class_b_clean))

class_a_sample = class_a_clean.sample(
    n=n,
    random_state=42
)

class_b_sample = class_b_clean.sample(
    n=n,
    random_state=42
)


data = pd.concat(
    [class_a_sample, class_b_sample],
    ignore_index=True
)

data = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print(data["label"].value_counts())

data.head()

label
1    234
0    234
Name: count, dtype: int64


,username,label
0,Yimmy702,1
1,BadB,1
2,Lux,1
3,simmjazeera,0
4,Bx1,1


#Testing Section

In [ ]:
X = data["username"]
y = data["label"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Training usernames:", len(X_train))
print("Testing usernames:", len(X_test))

Training usernames: 374
Testing usernames: 94


In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True
)


X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)


print("Number of features:", X_train_tfidf.shape[1])

Number of features: 6991


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")

Model trained successfully.


In [ ]:
predictions = model.predict(X_test_tfidf)


print("Accuracy:")
print(accuracy_score(y_test, predictions))


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=[
            "Class B",
            "Class A"
        ]
    )
)


print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

Accuracy:
0.6276595744680851

Classification Report:
              precision    recall  f1-score   support

     Class B       0.60      0.79      0.68        47
     Class A       0.69      0.47      0.56        47

    accuracy                           0.63        94
   macro avg       0.64      0.63      0.62        94
weighted avg       0.64      0.63      0.62        94


Confusion Matrix:
[[37 10]
 [25 22]]


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

import numpy as np

X = data["username"]
y = data["label"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
pr_auc_scores = []


fold = 1

for train_index, test_index in skf.split(X, y):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        lowercase=True
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)


    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train_tfidf,
        y_train
    )

    predictions = model.predict(
        X_test_tfidf
    )

    probabilities = model.predict_proba(
        X_test_tfidf
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )


    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    pr_auc_scores.append(pr_auc)

    print(f"\nFOLD {fold}")
    print("--------------------")

    print(
        "Accuracy:",
        round(accuracy, 3)
    )

    print(
        "Class A Precision:",
        round(precision, 3)
    )

    print(
        "Class A Recall:",
        round(recall, 3)
    )

    print(
        "Class A F1:",
        round(f1, 3)
    )

    print(
        "PR-AUC:",
        round(pr_auc, 3)
    )


    fold += 1

print("5-FOLD CROSS-VALIDATION RESULTS")
print("CLASS A vs CLASS B")


print(
    "Accuracy:",
    round(np.mean(accuracy_scores), 3),
    "+/-",
    round(np.std(accuracy_scores), 3)
)


print(
    "Class A Precision:",
    round(np.mean(precision_scores), 3),
    "+/-",
    round(np.std(precision_scores), 3)
)


print(
    "Class A Recall:",
    round(np.mean(recall_scores), 3),
    "+/-",
    round(np.std(recall_scores), 3)
)


print(
    "Class A F1:",
    round(np.mean(f1_scores), 3),
    "+/-",
    round(np.std(f1_scores), 3)
)


print(
    "PR-AUC:",
    round(np.mean(pr_auc_scores), 3),
    "+/-",
    round(np.std(pr_auc_scores), 3)
)


FOLD 1
--------------------
Accuracy: 0.798
Class A Precision: 0.804
Class A Recall: 0.787
Class A F1: 0.796
PR-AUC: 0.842

FOLD 2
--------------------
Accuracy: 0.617
Class A Precision: 0.617
Class A Recall: 0.617
Class A F1: 0.617
PR-AUC: 0.691

FOLD 3
--------------------
Accuracy: 0.617
Class A Precision: 0.657
Class A Recall: 0.489
Class A F1: 0.561
PR-AUC: 0.749

FOLD 4
--------------------
Accuracy: 0.71
Class A Precision: 0.794
Class A Recall: 0.574
Class A F1: 0.667
PR-AUC: 0.773

FOLD 5
--------------------
Accuracy: 0.677
Class A Precision: 0.722
Class A Recall: 0.565
Class A F1: 0.634
PR-AUC: 0.789

5-FOLD CROSS-VALIDATION RESULTS
CLASS A vs CLASS B
Accuracy: 0.684 +/- 0.067
Class A Precision: 0.719 +/- 0.074
Class A Recall: 0.607 +/- 0.099
Class A F1: 0.655 +/- 0.078
PR-AUC: 0.769 +/- 0.05


In [ ]:
# Restore the original fitted A/B classifier after cross-validation, which reuses
# `model` and `vectorizer` variable names within its folds.
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=True)
# Recreate the same 80/20 stratified training partition from the original testing cell.
_ab_X_train, _ab_X_test, _ab_y_train, _ab_y_test = train_test_split(
    data["username"].astype(str), data["label"], test_size=0.20,
    random_state=42, stratify=data["label"])
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(vectorizer.fit_transform(_ab_X_train), _ab_y_train)
print("Original A/B classifier restored for classify_username().")

In [ ]:
def classify_username(username):

    username_vector = vectorizer.transform([username])

    prediction = model.predict(username_vector)[0]

    probability = model.predict_proba(username_vector)[0][1]

    if prediction == 1:
        result = "Class A-like"
    else:
        result = "Class B-like"

    print("Username:", username)
    print("Prediction:", result)
    print(
        "Class A probability:",
        round(probability, 3)
    )

In [ ]:
print(classify_username('Craig_Levine'))

Username: Craig_Levine
Prediction: Class B-like
Class A probability: 0.472
None


## Supplement A — additional statistics for the original Class A/B experiment

The original Class A/B data loading, balancing, TF-IDF model, train/test split, cross-validation, and prediction cells above are retained **unchanged**. These supplementary tests compare group-level username features and evaluate classifier performance. Adjusted p-values account for multiple testing.


In [ ]:
# Descriptive features, hypothesis tests, multiple-testing correction, and effect sizes.
# Descriptive comparisons are observational associations, not causal or diagnostic evidence.
from scipy.stats import mannwhitneyu, fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests

feature_df = data[["username", "label"]].copy()
s = feature_df["username"]
feature_df["length"] = s.str.len()
feature_df["digit_count"] = s.str.count(r"\d")
feature_df["underscore_count"] = s.str.count("_")
feature_df["has_digit"] = s.str.contains(r"\d", regex=True)
feature_df["ends_digit"] = s.str.contains(r"\d$", regex=True)
feature_df["has_underscore"] = s.str.contains("_", regex=False)
feature_df["starts_letter"] = s.str.contains(r"^[A-Za-z]", regex=True)
feature_df["mixed_case"] = s.apply(lambda z: any(c.islower() for c in z) and any(c.isupper() for c in z))
feature_df["repeat_3plus"] = s.str.contains(r"(.)\1{2,}", regex=True)

rows = []
for feature in ["length", "digit_count", "underscore_count"]:
    a = feature_df.loc[feature_df.label == 1, feature].to_numpy()
    b = feature_df.loc[feature_df.label == 0, feature].to_numpy()
    stat, p = mannwhitneyu(a, b, alternative="two-sided", method="asymptotic")
    # Rank-biserial effect size, positive when Class A tends larger
    effect = 2 * stat / (len(a) * len(b)) - 1
    rows.append(dict(feature=feature, test="Mann–Whitney U", p=p,
                     A_value=float(np.median(a)), B_value=float(np.median(b)),
                     effect_rank_biserial=effect, odds_ratio=np.nan))
for feature in ["has_digit", "ends_digit", "has_underscore", "starts_letter", "mixed_case", "repeat_3plus"]:
    ct = pd.crosstab(feature_df["label"], feature_df[feature]).reindex(index=[0,1], columns=[False,True], fill_value=0)
    # Fisher's exact test works even with small cell counts
    odds, p = fisher_exact(ct.to_numpy())
    a_rate = ct.loc[1, True] / ct.loc[1].sum()
    b_rate = ct.loc[0, True] / ct.loc[0].sum()
    rows.append(dict(feature=feature, test="Fisher exact", p=p,
                     A_value=a_rate, B_value=b_rate,
                     effect_rank_biserial=np.nan, odds_ratio=odds))
feature_tests = pd.DataFrame(rows)
feature_tests["p_holm"] = multipletests(feature_tests["p"], method="holm")[1]
feature_tests["p_bh_fdr"] = multipletests(feature_tests["p"], method="fdr_bh")[1]
print("A_value/B_value: medians for counts, proportions for boolean features.")
print("Odds ratio compares Class A vs Class B for presence of the feature.")
display(feature_tests.sort_values("p_holm"))


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, log_loss,
                             brier_score_loss, matthews_corrcoef)
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(2,5), lowercase=True),
                    LogisticRegression(max_iter=1500, random_state=42))
X_all = data["username"].astype(str).reset_index(drop=True)
y_all = data["label"].astype(int).reset_index(drop=True)
# out-of-fold predictions, vectorizer fitting occurs INSIDE each fold
p_oof = cross_val_predict(clf, X_all, y_all, cv=cv, method="predict_proba")[:,1]
y_oof = (p_oof >= 0.5).astype(int)
base_pred = cross_val_predict(DummyClassifier(strategy="most_frequent"),
                              X_all.to_frame(), y_all, cv=cv, method="predict")
correct_model = y_oof == y_all.to_numpy()
correct_base = base_pred == y_all.to_numpy()
paired = np.array([[np.sum(correct_model & correct_base), np.sum(correct_model & ~correct_base)],
                   [np.sum(~correct_model & correct_base), np.sum(~correct_model & ~correct_base)]])
mc_p = mcnemar(paired, exact=(paired[0,1]+paired[1,0] < 25), correction=True).pvalue
accurate = int(correct_model.sum())
ci_low, ci_high = proportion_confint(accurate, len(y_all), alpha=.05, method="wilson")
score_table = pd.DataFrame([{
    "n":len(y_all), "accuracy":accuracy_score(y_all, y_oof),
    "accuracy_95CI_low":ci_low, "accuracy_95CI_high":ci_high,
    "balanced_accuracy":balanced_accuracy_score(y_all, y_oof),
    "macro_F1":f1_score(y_all, y_oof, average="macro"),
    "MCC":matthews_corrcoef(y_all, y_oof),
    "ROC_AUC":roc_auc_score(y_all, p_oof),
    "average_precision":average_precision_score(y_all, p_oof),
    "Brier_score":brier_score_loss(y_all, p_oof),
    "log_loss":log_loss(y_all, p_oof),
    "majority_baseline_accuracy":accuracy_score(y_all,base_pred),
    "McNemar_p_vs_baseline":mc_p
}])
display(score_table.T.rename(columns={0:"value"}))
print("McNemar's test evaluates paired out-of-fold correctness versus a majority-class baseline.")
print("Wilson interval describes pooled out-of-fold accuracy; observations may not be independent.")


## Supplement B — observed gender labels within Class B (X and YouTube)

**The Class A/B experiment above remains the main assignment.** This supplement uses the **actual Class B source CSV** (`Class_B_X_YouTube_Final(2).csv`) and its `handle`, `qid`, `platform`, and `gender` columns. It does not infer the gender of a person from a username or assume that undocumented categories are known. Recorded labels are metadata, not independently verified self-identification.

We report original raw labels and missingness, then use records with **exactly `male` or `female`** for a transparent, restricted two-category *exploratory* comparison. Other specified genders, trans labels, multi-valued labels, and missing values remain reported separately and are **not** automatically reassigned. Those exclusions limit the scope of the analysis and should be disclosed in the write-up. Because repeated QIDs can occur, statistical tests are performed on one observation per QID **within platform**; X and YouTube are analyzed separately to avoid mixing naming conventions.

The tests assess **group-level associations** in a nonrepresentative set of public figures. They are not a validated means of predicting any individual user's gender.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

# Colab users: the earlier Class B upload may already be the enriched file.
GENDER_CSV = None  # Set to a filename or complete path to override automatic detection.
if GENDER_CSV is None:
    candidate_paths = [Path('/mnt/data/Class_B_X_YouTube_Final(2).csv'),
                       Path('Class_B_X_YouTube_Final(2).csv'),
                       Path('Class_B_X_YouTube_Final(1).csv')]
    if 'class_b_file' in globals():
        candidate_paths.insert(0, Path(class_b_file))
    for candidate in candidate_paths:
        if candidate.exists():
            cols = set(pd.read_csv(candidate, nrows=0).columns.str.strip().str.lower())
            if {'handle', 'gender', 'qid', 'platform'}.issubset(cols):
                GENDER_CSV = str(candidate)
                break
if GENDER_CSV is None:
    try:
        from google.colab import files
        print('Upload Class_B_X_YouTube_Final(2).csv for Supplement B:')
        uploaded_gender = files.upload()
        GENDER_CSV = next(iter(uploaded_gender)) if uploaded_gender else None
    except ImportError:
        raise FileNotFoundError('Set GENDER_CSV to the path of your Class B source CSV.')
if not GENDER_CSV:
    raise FileNotFoundError('No gender CSV provided. Upload one or skip Supplement B.')
class_b_gender = pd.read_csv(GENDER_CSV, dtype={'qid': 'string', 'handle': 'string',
                                                  'platform': 'string', 'gender': 'string'},
                             low_memory=False)
class_b_gender.columns = class_b_gender.columns.str.strip().str.lower()
required = {'handle', 'gender', 'qid', 'platform'}
if not required.issubset(class_b_gender):
    raise ValueError(f'Expected columns {required}, found {set(class_b_gender.columns)}')
for col in ['qid', 'handle', 'platform', 'gender']:
    class_b_gender[col] = class_b_gender[col].astype('string').str.strip()
    class_b_gender.loc[class_b_gender[col].eq(''), col] = pd.NA
class_b_gender['gender_norm'] = class_b_gender['gender'].str.casefold()
print('Source:', GENDER_CSV)
print('Total Class B records:', len(class_b_gender))
print('Nonmissing recorded gender:', int(class_b_gender.gender_norm.notna().sum()))
print('Missing gender:', int(class_b_gender.gender_norm.isna().sum()))
print('Gender value distribution (unaltered source categories):')
display(class_b_gender['gender'].value_counts(dropna=False).rename_axis('gender').to_frame('count').head(60))
print('By platform:')
display(pd.crosstab(class_b_gender.platform, class_b_gender.gender_norm.notna(),
                    rownames=['platform'], colnames=['gender recorded'], margins=True))

In [ ]:
# Audit repeated identities / account handles and potential conflicting labels.
source = class_b_gender.dropna(subset=['qid', 'platform', 'handle']).copy()
conflicts = (source.dropna(subset=['gender_norm'])
             .groupby(['platform', 'qid'])['gender_norm'].nunique())
print('QID-within-platform groups containing conflicting nonmissing gender strings:',
      int((conflicts > 1).sum()))
print('Repeated QID-within-platform groups:',
      int(source.duplicated(['platform', 'qid']).sum()))
print('Repeated handle-within-platform rows:',
      int(source.duplicated(['platform', 'handle']).sum()))
# Keep only pairs without conflicting observed gender labels.
source = source.loc[~source.set_index(['platform', 'qid']).index.isin(
    conflicts[conflicts > 1].index)].copy()
# Prefer a row with a known label if an identity repeats.
source['has_gender'] = source.gender_norm.notna().astype(int)
source = (source.sort_values('has_gender', ascending=False, kind='stable')
          .drop_duplicates(['platform', 'qid'], keep='first').copy())
# Exact coding avoids treating trans women/men or multi-label strings as arbitrary 'other'.
source['comparison_gender'] = source.gender_norm.where(source.gender_norm.isin(['male', 'female']))
print('Deduplicated QID-platform observations:', len(source))
print('Restricted comparison labels:')
display(pd.crosstab(source.platform, source.comparison_gender, margins=True))
print('Excluded from restricted binary tests (other stated or missing):',
      int(source.comparison_gender.isna().sum()))

# Use the same family of orthographic features as the core project.
def gender_username_features(s):
    s = str(s)
    chars = [c for c in s if c.isalpha()]
    return {'length': len(s),
            'digit_count': sum(c.isdigit() for c in s),
            'punct_count': sum(not c.isalnum() for c in s),
            'has_digit': int(any(c.isdigit() for c in s)),
            'ends_digit': int(bool(re.search(r'\d$', s))),
            'has_underscore': int('_' in s),
            'has_uppercase': int(any(c.isupper() for c in chars)),
            'mixed_case': int(any(c.islower() for c in chars) and any(c.isupper() for c in chars)),
            'has_year_like_number': int(bool(re.search(r'(?<!\d)(19\d{2}|20\d{2})(?!\d)', s))),
            'long_repetition': int(bool(re.search(r'(.)\1{2,}', s.lower())))}
comparison = source.dropna(subset=['comparison_gender']).copy()
feat = pd.DataFrame([gender_username_features(s) for s in comparison.handle],
                    index=comparison.index)
comparison = pd.concat([comparison, feat], axis=1)
print('Descriptive statistics by platform and recorded binary category:')
display(comparison.groupby(['platform', 'comparison_gender'])
        [['length', 'digit_count', 'punct_count', 'has_digit',
          'has_underscore', 'mixed_case']].agg(['count', 'mean', 'median']).round(3))

In [ ]:
# Two-group comparisons within each platform; effect sizes and Holm corrections.
# No tests are performed for sparsely observed gender strings.
NUMERIC = ['length', 'digit_count', 'punct_count']
BINARY = ['has_digit', 'ends_digit', 'has_underscore', 'has_uppercase',
          'mixed_case', 'has_year_like_number', 'long_repetition']
results = []
for platform, part in comparison.groupby('platform'):
    male = part.loc[part.comparison_gender.eq('male')]
    female = part.loc[part.comparison_gender.eq('female')]
    if min(len(male), len(female)) < 2:
        print('Insufficient observations on', platform)
        continue
    for variable in NUMERIC:
        if part[variable].nunique() < 2:
            continue
        stat, p = mannwhitneyu(male[variable], female[variable], alternative='two-sided')
        # Common-language superiority 2U/(n1*n2)-1; positive indicates male > female.
        effect = 2 * stat / (len(male) * len(female)) - 1
        results.append(dict(platform=platform, feature=variable, test='Mann–Whitney U',
                            statistic=stat, p=p, effect_name='rank-biserial r', effect=effect,
                            male_rate_or_mean=male[variable].mean(),
                            female_rate_or_mean=female[variable].mean()))
    for variable in BINARY:
        tab = pd.crosstab(part.comparison_gender, part[variable]).reindex(
            index=['male', 'female'], columns=[0, 1], fill_value=0)
        if tab[1].sum() == 0 or tab[0].sum() == 0:
            continue
        if (tab.to_numpy() < 5).any():
            stat, p = fisher_exact(tab.to_numpy())
            method = 'Fisher exact'
            effect = float(stat)
            effect_name = 'odds ratio (male: female)'
        else:
            stat, p, _, _ = chi2_contingency(tab.to_numpy(), correction=False)
            method = 'Chi-square'
            # Signed phi: positive means feature more common among male records.
            effect = np.sign(male[variable].mean() - female[variable].mean()) * np.sqrt(stat / tab.to_numpy().sum())
            effect_name = 'signed phi'
        results.append(dict(platform=platform, feature=variable, test=method,
                            statistic=stat, p=p, effect_name=effect_name, effect=effect,
                            male_rate_or_mean=male[variable].mean(),
                            female_rate_or_mean=female[variable].mean()))
gender_tests = pd.DataFrame(results)
if gender_tests.empty:
    print('No comparisons possible.')
else:
    gender_tests['p_holm'] = multipletests(gender_tests.p, method='holm')[1]
    gender_tests['p_bh'] = multipletests(gender_tests.p, method='fdr_bh')[1]
    gender_tests['significant_holm_05'] = gender_tests.p_holm < 0.05
    display(gender_tests.sort_values(['p_holm', 'platform', 'feature']).round(5))
    gender_tests.to_csv('Class_B_Gender_Supplement_Statistics.csv', index=False)
    print('Saved: Class_B_Gender_Supplement_Statistics.csv')

# Descriptive plot, not an individual gender predictor.
axes_data = comparison.groupby(['platform','comparison_gender']).length.median().unstack()
axes_data.plot(kind='bar', rot=0, ylabel='Median username length',
               title='Class B username length by recorded category and platform')
plt.tight_layout()
plt.show()

### Interpretation / limitations for your assignment

- **Primary research outcome:** the A/B classifier above, unchanged in purpose; Supplement A adds performance diagnostics and tests.
- **Supplement B:** descriptive and inferential associations between recorded gender labels and username form, **within** X and YouTube. Labels are from the supplied Class B file, not generated guesses. This analysis deliberately does **not** predict an individual's gender.
- **Restricted sample:** only records labeled exactly `male` or `female` enter the two-category tests; report exclusions and avoid generalizations to other identities. Missing labels and other recorded values are displayed but not reclassified.
- **Dependence and selection:** repeated QIDs within each platform are collapsed, but shared users across platforms and uncertainty in public-figure identifiers can still violate independence. Results should be described as exploratory, not representative of all users or proof of causal gender differences.
- **Multiplicity:** adjusted p-values and effect sizes matter more than isolated significant raw p-values. Report the correction family, actual denominators, and direction of effects. Extremely large sample sizes can make substantively small effects statistically significant.
- **Reproducibility:** keep the original dataset, note the retrieval/label source and date, and report unresolved metadata-quality limitations in any write-up.